<a href="https://colab.research.google.com/github/Syeda-Rb-47/Building-GlueScore--An-Interpretable-Glue-Likeness-Scoring-Pipeline/blob/main/1_Dataset_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTS

In [ ]:
# Installing required libraries and importing modules.
!pip install rdkit requests

import pandas as pd
import requests
from rdkit import Chem
from rdkit.Chem import Descriptors

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 33.3 MB/s eta 0:00:00
Mounted at /content/drive


STANDARDIZING AND FILTERING FUNCTIONS

In [ ]:
# Function to canonicalize SMILES using RDKit
def to_canonical_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None

# Function to remove inorganics
def is_organic(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    return any(atom.GetAtomicNum() == 6 for atom in mol.GetAtoms())

# Function to get number of Ro5 violations
def get_ro5_violations(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 99
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    return sum([mw > 500, logp > 5, hbd > 5, hba > 10])

WORKING WITH MOLECULAR GLUE DATASET

In [ ]:
# Loading the MolGlueDB dataset from Drive.
df = pd.read_csv("/content/drive/MyDrive/Thesis Datasets/MolGlueDB_full.csv")
df.head()

,DATAID,SMILES,Name,IUPAC,StdInChl,StdInChIKey,Pharmacophore,Core,ResearchStage,TherapeuticUsage,...,RotatableBondCount,Rings,AliphaticRings,AromaticRings,AliphaticHeteroRings,AromaticHeteroRings,HeavyAtoms,HeteroAtoms,SpiroAtoms,BridgeheadAtoms
0,1,[2H]C([2H])([2H])OC1=CC=C2C=C(C(=O)NC3=CC=C4C(...,36,"N-[2-(2,6-dioxopiperidin-3-yl)-1-oxo-3H-isoind...",InChI=1S/C25H21N3O5/c1-33-19-6-4-14-10-16(3-2-...,JGWFFKIYBDZDFI-FIBGUPNXSA-N,Glutarimide,LenalidomideType,Discovery,NaN,...,5,5,2,3,2,0,33,8,0,0
1,2,CC/C=C\C[C@@H]1C(=O)CC[C@@H]1CC(=O)N[C@H](C(=O...,"(3R,7S)-JA-L-Ile","(2S,3S)-3-methyl-2-[[2-[(1R,2S)-3-oxo-2-[(Z)-p...",InChI=1S/C18H29NO4/c1-4-6-7-8-14-13(9-10-15(14...,IBZYPBGPOGJMBF-QPERPISQSA-N,miscellaneous,Non-domideType,Discovery,NaN,...,9,1,1,0,0,0,23,5,0,0
2,3,C#CC1=CC=C(NC(=O)OC[C@]23CN(S(=O)(=O)C4=C(C)C=...,iDeg-6,"[(1R,2S,6S,8S)-4-(4-tert-butyl-2-methylphenyl)...",InChI=1S/C32H40N2O4S/c1-8-22-9-12-26(13-10-22)...,SVVLXZSOWBLJBW-YUGNUMPCSA-N,miscellaneous,Non-domideType,Discovery,NaN,...,5,6,4,2,1,0,39,7,0,2
3,4,C#CC1=CC=C(NC(=O)OC[C@]23CN(S(=O)(=O)C4=CC=C(C...,iDeg-3,"[(1R,2S,6S,8S)-4-(4-tert-butylphenyl)sulfonyl-...",InChI=1S/C31H38N2O4S/c1-7-21-8-12-25(13-9-21)3...,LPVXGYZMYNYSRZ-GGWKSQAJSA-N,miscellaneous,Non-domideType,Discovery,NaN,...,5,6,4,2,1,0,38,7,0,2
4,5,O=C1CCC(NC(=O)C2=C(F)C=C([N+](=O)[O-])C=C2)C(=...,7d,"N-(2,6-dioxopiperidin-3-yl)-2-fluoro-4-nitrobe...",InChI=1S/C12H10FN3O5/c13-8-5-6(16(20)21)1-2-7(...,CSOFOICJBYMWDE-UHFFFAOYSA-N,miscellaneous,Non-domideType,Discovery,NaN,...,3,2,1,1,1,0,21,9,0,0


In [ ]:
# Keeping only relevant columns and renaming to match standard format.
df = df[["DATAID", "SMILES", "StdInChIKey"]].copy()
df = df.rename(columns={
    "DATAID":     "Source_ID",
    "SMILES":     "Canonical_SMILES"
})

n_glue_raw   = len(df)
print("Collected from MolGlueDB:", n_glue_raw)

Collected from MolGlueDB: 1840


In [ ]:
# Standardizing SMILES using RDKit and removing invalid entries.
before = len(df)
df["Canonical_SMILES"] = df["Canonical_SMILES"].apply(to_canonical_smiles)
df = df[df["Canonical_SMILES"].notna()].copy()
print("After RDKit canonicalization:", len(df), " (lost", before - len(df), ")")

# Removing duplicate entries.
before = len(df)
df = df.drop_duplicates(subset=["StdInChIKey"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["Canonical_SMILES"]).reset_index(drop=True)
print("After deduplication:", len(df), " (lost", before - len(df), ")")

# Assigning label.
df["Label"] = 1

print("Final glue set shape:", df.shape)
df.head()

After RDKit canonicalization: 1840  (lost 0 )
After deduplication: 1629  (lost 211 )
Final glue set shape: (1629, 4)


,Source_ID,Canonical_SMILES,StdInChIKey,Label
0,1,[2H]C([2H])([2H])Oc1ccc2cc(C(=O)Nc3ccc4c(c3)CN...,JGWFFKIYBDZDFI-FIBGUPNXSA-N,1
1,2,CC/C=C\C[C@@H]1C(=O)CC[C@@H]1CC(=O)N[C@H](C(=O...,IBZYPBGPOGJMBF-QPERPISQSA-N,1
2,3,C#Cc1ccc(NC(=O)OC[C@]23CN(S(=O)(=O)c4ccc(C(C)(...,SVVLXZSOWBLJBW-YUGNUMPCSA-N,1
3,4,C#Cc1ccc(NC(=O)OC[C@]23CN(S(=O)(=O)c4ccc(C(C)(...,LPVXGYZMYNYSRZ-GGWKSQAJSA-N,1
4,5,O=C1CCC(NC(=O)c2ccc([N+](=O)[O-])cc2F)C(=O)N1,CSOFOICJBYMWDE-UHFFFAOYSA-N,1


In [ ]:
# Saving the cleaned DataFrame to Google Drive.
df.to_csv("/content/drive/MyDrive/Thesis Datasets/MolGlueDB dataset.csv", index=False)

WORKING WITH CHEMBL BACKGROUND SET

In [ ]:
# Fetching small molecules from the ChEMBL API using pagination.
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data/molecule.json"

TARGET_N = 3000
LIMIT    = 1000

# Checking total molecules available first.
for attempt in range(3):
    try:
        r = requests.get(BASE_URL, params={"max_phase__gte": 1, "limit": 1}, timeout=60)
        r.raise_for_status()
        FETCH_N = r.json()["page_meta"]["total_count"]
        print("Total available in ChEMBL:", FETCH_N)
        break
    except (requests.exceptions.Timeout, requests.exceptions.HTTPError, requests.exceptions.JSONDecodeError):
        print("Attempt", attempt + 1, "failed, retrying...")
        if attempt == 2:
            raise

Total available in ChEMBL: 16784


In [ ]:
# Parsing and storing structural identifiers.
rows   = []
offset = 0

while len(rows) < FETCH_N:
    params = {
        "max_phase__gte": 1,
        "order_by":       "molecule_chembl_id",
        "limit":          LIMIT,
        "offset":         offset,
    }

    for attempt in range(1000):
        try:
            r = requests.get(BASE_URL, params=params, timeout=120)
            r.raise_for_status()
            break
        except (requests.exceptions.Timeout, requests.exceptions.HTTPError):
            print("Timeout on attempt", attempt + 1, ", retrying...")
            if attempt == 999:
                raise

    data = r.json()

    molecules = data.get("molecules", [])
    if not molecules:
        break

    for m in molecules:
        s = m.get("molecule_structures") or {}

        chembl_id        = m.get("molecule_chembl_id")
        smiles           = s.get("canonical_smiles")
        inchikey         = s.get("standard_inchi_key")
        therapeutic_flag = m.get("therapeutic_flag")
        molecule_type    = m.get("molecule_type")

        if chembl_id and smiles and inchikey:
            rows.append({
                "ChemblID":         chembl_id,
                "Canonical_SMILES": smiles,
                "StdInChIKey":      inchikey,
                "therapeutic_flag": therapeutic_flag,
                "molecule_type":    molecule_type,
            })

    offset += LIMIT
    print("Fetched so far:", len(rows))

n_bg_raw = len(rows)
print("Collected:", len(rows))

Fetched so far: 965
Fetched so far: 1741
Fetched so far: 2732
Fetched so far: 3595
Fetched so far: 4593
Fetched so far: 5592
Fetched so far: 6148
Fetched so far: 6545
Fetched so far: 7519
Fetched so far: 8313
Fetched so far: 9152
Fetched so far: 9866
Fetched so far: 10379
Fetched so far: 11050
Fetched so far: 11556
Fetched so far: 12091
Fetched so far: 12785
Collected: 12785


In [ ]:
# Keeping only small molecules with therapeutic intent.
df = pd.DataFrame(rows)
print("Raw from API:", len(df))

# 1. Molecule type filter
before = len(df)
df = df[df["molecule_type"] == "Small molecule"].copy()
print("After molecule_type filter:", len(df), " (lost", before - len(df), ")")

# 2. Therapeutic flag filter
before = len(df)
df = df[df["therapeutic_flag"] == True].copy()
df = df.drop(columns=["therapeutic_flag", "molecule_type"])
df = df.reset_index(drop=True)
print("After therapeutic_flag filter:", len(df), " (lost", before - len(df), ")")

# Standardizing SMILES using RDKit and removing invalid entries.
before = len(df)
df["Canonical_SMILES"] = df["Canonical_SMILES"].apply(to_canonical_smiles)
df = df[df["Canonical_SMILES"].notna()].copy()
print("After RDKit canonicalization:", len(df), " (lost", before - len(df), ")")

# Removing inorganics
before = len(df)
df = df[df["Canonical_SMILES"].apply(is_organic)].copy()
df = df.reset_index(drop=True)
print("After is_organic filter:", len(df), " (lost", before - len(df), ")")

# Keeping molecules with Ro5 violations <= 2 (bRo5)
before = len(df)
df = df[df["Canonical_SMILES"].apply(get_ro5_violations) <= 2].copy()
df = df.reset_index(drop=True)
print("After Ro5 <= 2 filter:", len(df), " (lost", before - len(df), ")")

# Removing duplicates
before = len(df)
df = df.drop_duplicates(subset=["StdInChIKey"])
df = df.drop_duplicates(subset=["Canonical_SMILES"])
df = df.reset_index(drop=True)
print("After deduplication:", len(df), " (lost", before - len(df), ")")

# Trimming to TARGET_N if needed
before = len(df)
if len(df) > TARGET_N:
    df = df.iloc[:TARGET_N]
print("After trim to TARGET_N:", len(df), " (lost", before - len(df), ")")

# Renaming ID column and assigning Label
df = df.rename(columns={"ChemblID": "Source_ID"})
df["Label"] = 0

print("Final background set shape:", df.shape)
df.head()

Raw from API: 12785
After molecule_type filter: 12391  (lost 394 )
After therapeutic_flag filter: 3026  (lost 9365 )
After RDKit canonicalization: 3026  (lost 0 )
After is_organic filter: 3001  (lost 25 )
After Ro5 <= 2 filter: 2875  (lost 126 )
After deduplication: 2875  (lost 0 )
After trim to TARGET_N: 2875  (lost 0 )
Final background set shape: (2875, 4)


,Source_ID,Canonical_SMILES,StdInChIKey,Label
0,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,ZKLPARSLTMPFCP-UHFFFAOYSA-N,0
1,CHEMBL100116,CC(C)=CCN1CCC2(C)c3cc(O)ccc3CC1C2C,VOKSWYLNZZRQPF-UHFFFAOYSA-N,0
2,CHEMBL1002,CC(C)(C)NC[C@H](O)c1ccc(O)c(CO)c1,NDAUXUAQIAJITI-LBPRGKRZSA-N,0
3,CHEMBL1003,O=C([O-])[C@H]1/C(=C/CO)O[C@@H]2CC(=O)N21.[K+],ABVRVIZBZKUTMK-JSYANWSFSA-M,0
4,CHEMBL1004,CN(C)CCOC(C)(c1ccccc1)c1ccccn1,HCFDWZZGGLSKEP-UHFFFAOYSA-N,0


In [ ]:
# Saving the cleaned DataFrame to Google Drive.
df.to_csv("/content/drive/MyDrive/Thesis Datasets/ChEMBL dataset.csv", index=False)

COMPARING BACKGROUND AND GLUE DATASETS FOR Ro5 VIOLATIONS

In [ ]:
df_glue = pd.read_csv("/content/drive/MyDrive/Thesis Datasets/MolGlueDB dataset.csv")
df_bg   = pd.read_csv("/content/drive/MyDrive/Thesis Datasets/ChEMBL dataset.csv")

In [ ]:
# Calculating Ro5 violations for both sets
df_glue["Ro5_violations"] = df_glue["Canonical_SMILES"].apply(get_ro5_violations)
df_bg["Ro5_violations"]   = df_bg["Canonical_SMILES"].apply(get_ro5_violations)

# Comparing distributions
glue_dist = df_glue["Ro5_violations"].value_counts().sort_index()
bg_dist   = df_bg["Ro5_violations"].value_counts().sort_index()

comparison = pd.DataFrame({
    "Ro5 Violations": [0, 1, 2, "3+"],
    "Mol Glue (n)":   [
        (df_glue["Ro5_violations"] == 0).sum(),
        (df_glue["Ro5_violations"] == 1).sum(),
        (df_glue["Ro5_violations"] == 2).sum(),
        (df_glue["Ro5_violations"] >= 3).sum(),
    ],
    "Background (n)": [
        (df_bg["Ro5_violations"] == 0).sum(),
        (df_bg["Ro5_violations"] == 1).sum(),
        (df_bg["Ro5_violations"] == 2).sum(),
        (df_bg["Ro5_violations"] >= 3).sum(),
    ],
})

comparison["Mol Glue (%)"]   = (comparison["Mol Glue (n)"]   / len(df_glue) * 100).round(1)
comparison["Background (%)"] = (comparison["Background (n)"] / len(df_bg)   * 100).round(1)

print(comparison.to_string(index=False))

Ro5 Violations  Mol Glue (n)  Background (n)  Mol Glue (%)  Background (%)
             0          1088            2068          66.8            71.9
             1           315             513          19.3            17.8
             2           169             294          10.4            10.2
            3+            57               0           3.5             0.0
